# 02 · Parse and clean

1. **Parse** loads everything in `data/raw` into the `core.*` tables. It's safe to re-run: existing rows are updated, never duplicated.
2. **Clean** flags every review (`clean.review_flags`) and adds a row per rule to `meta.cleaning_report`. Language detection takes a few minutes for hundreds of thousands of reviews.

Rules are in `src/clean/rules.py`, and thresholds in `configs/config.yaml` → `cleaning`. The results are explored in `03_eda.ipynb`.

In [1]:
import logging

from src.clean.run import clean_reviews
from src.collect.run import parse
from src.db.connection import init_db

logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
init_db()

## Parse

In [2]:
parse()  # rows loaded per source

,rows
store_details,28
reviews,833532
news,1073
player_counts,3311


## Clean

In [3]:
report = clean_reviews()
totals = report.groupby("rule")[["n_flagged", "n_total"]].sum()
totals["share"] = (totals["n_flagged"] / totals["n_total"]).round(3)
totals

,n_flagged,n_total,share
rule,,,
dropped,32047,833486,0.038
duplicate_text,15,833486,0.000
edited,121924,833486,0.146
empty,21045,833486,0.025
junk,23916,833486,0.029
lang_mismatch,6903,833486,0.008


`dropped` counts reviews with `keep = False`. Notebook 03, section 3, shows examples for each rule.